# mBERT (`bert-base-multilingual-cased`), paper-aligned fine-tuning
**Runtime > Change runtime type > GPU.**

Paper settings used (Sec. 4.4): AdamW, lr 2e-5, weight decay 0.01, linear schedule with 10% warmup, effective batch size 32 (16 x 2 accumulation to fit a T4), up to 15 epochs, early stopping patience 3, max length 512, seeds 42/123/2025 (mean ± std).

Early stopping/best epoch is chosen on validation macro-F1 (paper does not say which metric). **Runtime is long** (up to 15 epochs x 3 seeds). To go faster, set `SEEDS = [42]` or `MAX_LEN = 256`.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, numpy as np, pandas as pd

# ---- EDIT: Drive folder that holds gujarati_fake_real_combined.jsonl and the split files ----
DATA_DIR = "/content/drive/MyDrive/RM_24DCS041_054"
MODEL_DIR = f"{DATA_DIR}/models_paper"      # separate folder so earlier models are not overwritten
os.makedirs(MODEL_DIR, exist_ok=True)

SEEDS = [42, 123, 2025]   # paper (Sec. 4.4/5.1): three seeds, results reported as mean +/- std

In [ ]:
def load_split(name):
    d = pd.read_json(f"{DATA_DIR}/{name}.jsonl", lines=True, dtype={"news_id": str})
    assert {"news_id", "text", "label"} <= set(d.columns)
    d["text"] = d["text"].astype(str)
    d["label"] = d["label"].astype(str).str.strip().str.lower()
    assert d["label"].isin(["fake", "real"]).all()
    return d

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")

# one label encoding for every model: real = 0, fake = 1
label2id = {"real": 0, "fake": 1}
id2label = {0: "real", 1: "fake"}

X_train, y_train = train_df["text"].values, train_df["label"].map(label2id).values
X_val,   y_val   = val_df["text"].values,   val_df["label"].map(label2id).values
X_test,  y_test  = test_df["text"].values,  test_df["label"].map(label2id).values

n = len(train_df) + len(val_df) + len(test_df)
print(f"train {len(train_df)} ({len(train_df)/n:.1%}) | val {len(val_df)} ({len(val_df)/n:.1%}) | test {len(test_df)} ({len(test_df)/n:.1%})")
print("Expected ~70/15/15. If not, re-run 01_data_split.ipynb first.")
print(train_df["label"].value_counts().to_dict())

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

def evaluate(y_true, y_pred, n_boot=1000, seed=0):
    """Macro metrics + bootstrap 95% CI for macro-F1 (paper Sec. 5.1)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        boots.append(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p,
            "macro_recall": r, "macro_f1": f, "f1_ci_low": lo, "f1_ci_high": hi}

In [ ]:
import torch, shutil
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, EarlyStoppingCallback, set_seed)

CKPT = "bert-base-multilingual-cased"
NAME = "08_mbert"
MAX_LEN = 512

tok = AutoTokenizer.from_pretrained(CKPT)

def to_ds(df):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "label": df["label"].map(label2id).tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, preds), "macro_precision": pr, "macro_recall": rc, "macro_f1": f1}

In [ ]:
results, all_preds, best_val = [], {}, -1.0
save_path = f"{MODEL_DIR}/{NAME}_best"

for seed in SEEDS:
    set_seed(seed)   # also seeds the new classification head
    model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=f"/content/ckpts/{NAME}_seed{seed}",
        num_train_epochs=15,
        per_device_train_batch_size=16, gradient_accumulation_steps=2,   # effective batch 32
        per_device_eval_batch_size=64,
        learning_rate=2e-5, weight_decay=0.01,
        warmup_ratio=0.1, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=torch.cuda.is_available(), seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()

    val_f1 = trainer.evaluate()["eval_macro_f1"]
    pred = trainer.predict(test_ds).predictions.argmax(-1)
    m = evaluate(y_test, pred)
    m.update(seed=seed, val_macro_f1=val_f1)
    results.append(m); all_preds[seed] = pred
    print(f"seed {seed}: val macro-F1 {val_f1:.4f} | test macro-F1 {m['macro_f1']:.4f}")

    if val_f1 > best_val:                       # keep only the best seed's weights on Drive
        best_val = val_f1
        trainer.save_model(save_path); tok.save_pretrained(save_path)
        print("  saved new best model ->", save_path)

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/ckpts/{NAME}_seed{seed}", ignore_errors=True)

In [ ]:
res = pd.DataFrame(results).set_index("seed")
display(res.round(4))
cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
summ = res[cols].agg(["mean", "std"]).T
print("=== TEST RESULTS, mean +/- std over seeds " + str(list(res.index)) + " ===")
for c in cols:
    print(f"{c:16s}: {summ.loc[c, 'mean']:.4f} +/- {summ.loc[c, 'std']:.4f}")
res.to_csv(f"{MODEL_DIR}/08_mbert_results.csv")

first = SEEDS[0]
print(f"\nClassification report (seed {first}):")
print(classification_report(y_test, all_preds[first], target_names=["real", "fake"], digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, all_preds[first], display_labels=["real", "fake"], cmap="Blues")
plt.title(f"08_mbert - confusion matrix (seed {first})"); plt.show()